# Depth Estimation
`capabilities/depth/depth_estimator.py`  
`capabilities/depth/midas_depth.py`

Infer a per-pixel depth map from a **single image** — no stereo camera, no LiDAR.

| Class | Model | Speed (CPU) | Size | Best for |
|-------|-------|-------------|------|----------|
| `DepthEstimator` | Depth Anything V2 Small | ~5s | 100 MB | General use, **default** |
| `MiDaSDepth` | MiDaS DPT-Hybrid | ~8s | 400 MB | Unusual viewpoints, fallback |

Both return identical `DepthResult` — swap freely.

**Primary use case in this project:** Use Case B — plant height estimation  
→ depth of plant tip vs depth of soil base → height proportional to depth difference.

## Setup

In [ ]:
import sys, os, tempfile
sys.path.insert(0, os.path.abspath('../..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from capabilities.depth.depth_estimator import DepthEstimator
from capabilities.depth.midas_depth import MiDaSDepth
from sprout_detection.utils.image_gen import make_sprout_image

tmp = tempfile.mkdtemp()
plant_path = os.path.join(tmp, 'plant.png')
cv2.imwrite(plant_path, make_sprout_image(seed=42))

estimator = DepthEstimator()   # lazy-loads on first call
print('DepthEstimator ready (model loads on first estimate() call)')

## Basic usage — estimate a depth map

In [ ]:
result = estimator.estimate(plant_path)

print(f'Depth map shape : {result.depth_map.shape}')
print(f'Mode            : {result.depth_mode}  (0 = close, 1 = far)')
print(f'Depth range     : [{result.depth_min:.3f}, {result.depth_max:.3f}]')
print(f'Mean depth      : {result.depth_mean:.3f}')
print(f'Image size      : {result.image_width} × {result.image_height}')
print(f'Inference time  : {result.duration_ms:.0f} ms')

## Visualise the depth map

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Original image
axes[0].imshow(cv2.cvtColor(cv2.imread(plant_path), cv2.COLOR_BGR2RGB))
axes[0].set_title('Original Image', fontsize=11)

# Depth map (viridis — purple=close, yellow=far)
if result.depth_map is not None:
    im = axes[1].imshow(result.depth_map, cmap='viridis')
    plt.colorbar(im, ax=axes[1], label='Depth (0=close, 1=far)')
    axes[1].set_title('Depth Map (Depth Anything V2)', fontsize=11)
    
    # Pseudo-3D colour rendering
    depth_coloured = cm.plasma(result.depth_map)[:, :, :3]
    axes[2].imshow(depth_coloured)
    axes[2].set_title('Depth — Plasma Colormap', fontsize=11)
else:
    for ax in axes[1:]:
        ax.text(0.5, 0.5, 'Run with real model\nto see depth maps',
                transform=ax.transAxes, ha='center', va='center', color='gray')

for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## sample_point() — depth at a specific pixel

In [ ]:
img = cv2.imread(plant_path)
h, w = img.shape[:2]

# Sample points along a vertical transect through the plant
cx = w // 2   # centre column
sample_ys = [int(h * f) for f in [0.05, 0.15, 0.30, 0.50, 0.70, 0.85, 0.95]]
sample_labels = ['sky/top', 'plant tip', 'upper stem', 'mid stem', 'lower stem', 'soil surface', 'pot base']

print('Depth transect (centre column):')
depths = []
for y, label in zip(sample_ys, sample_labels):
    d = result.sample_point(cx, y)
    depths.append(d)
    bar = '█' * int(d * 25)
    print(f'  y={y:3d}  {label:<15} depth={d:.3f}  {bar}')

# Plot transect
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(sample_labels, depths, color='#3498db', alpha=0.85)
ax.set_xlabel('Depth value (0=close, 1=far)')
ax.set_title('Vertical Depth Transect', fontsize=11)
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## estimate_height() — Use Case B core computation

In [ ]:
# Define tip (top of plant) and base (soil surface)
tip_point  = (cx, sample_ys[1])   # plant tip
base_point = (cx, sample_ys[5])   # soil surface

height_info = result.estimate_height(tip_point, base_point)

print('Height estimation:')
print(f'  Tip depth   : {height_info["tip_depth"]:.4f}  (closer = lower value)')
print(f'  Base depth  : {height_info["base_depth"]:.4f}')
print(f'  Difference  : {height_info["depth_difference"]:.4f}  (proxy for height)')
print(f'  Mode        : {height_info["mode"]}')
print()
print('Note: In relative mode, depth_difference is a unitless proxy.')
print('To convert to cm, calibrate with a reference object of known height.')

# Visualise the measurement points on the image
if result.depth_map is not None:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(cv2.cvtColor(cv2.imread(plant_path), cv2.COLOR_BGR2RGB))
    axes[0].plot(*tip_point,  'r^', markersize=12, label=f'Tip (depth={height_info["tip_depth"]:.3f})')
    axes[0].plot(*base_point, 'bs', markersize=12, label=f'Base (depth={height_info["base_depth"]:.3f})')
    axes[0].legend(fontsize=8)
    axes[0].set_title('Measurement Points', fontsize=10)
    axes[0].axis('off')
    
    axes[1].imshow(result.depth_map, cmap='viridis')
    axes[1].plot(*tip_point,  'r^', markersize=12)
    axes[1].plot(*base_point, 'bs', markersize=12)
    axes[1].set_title(f'Depth Map — Δ={height_info["depth_difference"]:.3f}', fontsize=10)
    axes[1].axis('off')
    plt.tight_layout(); plt.show()

## region_stats() — depth statistics within a mask

In [ ]:
from capabilities.segmentation.hsv_segmentor import HSVSegmentor

seg = HSVSegmentor()
seg_result = seg.segment(plant_path, profile='green_plant')

if result.depth_map is not None and seg_result.mask is not None:
    # Resize mask to match depth map if needed
    mask_resized = cv2.resize(seg_result.mask,
                               (result.image_width, result.image_height),
                               interpolation=cv2.INTER_NEAREST)
    plant_mask = mask_resized.astype(bool)
    background_mask = ~plant_mask

    plant_stats = result.region_stats(plant_mask)
    bg_stats    = result.region_stats(background_mask)

    print('Depth stats — plant region vs background:')
    print(f'{"":<12} {"Plant region":>14}  {"Background":>12}')
    for key in ['mean', 'median', 'min', 'max', 'std']:
        print(f'  {key:<10} {plant_stats[key]:>12.4f}  {bg_stats[key]:>12.4f}')
    print(f'  {"pixels":<10} {plant_stats["pixel_count"]:>12d}  {bg_stats["pixel_count"]:>12d}')
else:
    print('Run with real models to see region stats')

## MiDaSDepth — identical interface, different backbone

In [ ]:
midas = MiDaSDepth()

result_midas = midas.estimate(plant_path)

print(f'MiDaS depth map shape: {result_midas.depth_map.shape}')
print(f'Depth range          : [{result_midas.depth_min:.3f}, {result_midas.depth_max:.3f}]')
print(f'Inference time       : {result_midas.duration_ms:.0f} ms')

# Same API as DepthEstimator — completely interchangeable
height_midas = result_midas.estimate_height(tip_point, base_point)
print(f'Height difference (MiDaS): {height_midas["depth_difference"]:.4f}')

## estimate_with_mask() — convenience method

In [ ]:
# Convenience: estimate depth AND compute region stats in one call
if seg_result.mask is not None:
    depth_result, stats = estimator.estimate_with_mask(
        plant_path,
        mask=seg_result.mask.astype(bool)
    )
    print('estimate_with_mask() — plant region stats:')
    import json
    print(json.dumps(stats, indent=2))